In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import osmnx as ox

In [2]:
DATA_BASE_PATH = Path("./data").resolve().absolute()
OSRM_BASE_URL = "http://localhost:5000"
GRAPH_PATH = DATA_BASE_PATH / "newson_krumm_reconstructed.osm.xml"

In [3]:
gps_data_path = DATA_BASE_PATH / "gps_data.parquet"
ground_truth_path = DATA_BASE_PATH / "ground_truth_route.parquet"
newson_krumm_route_network_path = DATA_BASE_PATH / "road_network.parquet"

In [4]:
ground_truth_df = duckdb.query(
    f"""
    WITH gt AS (
        SELECT edge_id, traversed, ROW_NUMBER() OVER () as row_num
        FROM '{ground_truth_path}'
    )
    SELECT gt.edge_id - 883000000000 AS edge_id , traversed, linestring
    FROM gt
    LEFT JOIN '{newson_krumm_route_network_path}' nkr 
    ON gt.edge_id = nkr.edge_id
    ORDER BY gt.row_num
    """
).to_df()
ground_truth_df

,edge_id,traversed,linestring
0,1147800801,1,"LINESTRING(-122.109748721123 47.6673012971878,..."
1,1147800802,1,"LINESTRING(-122.105398178101 47.6675292849541,..."
2,1147800421,1,"LINESTRING(-122.102048099041 47.6676607131958,..."
3,1147800422,1,"LINESTRING(-122.1028393507 47.6681300997734, -..."
4,1147800423,1,"LINESTRING(-122.103689610958 47.6685512065887,..."
...,...,...,...
571,1147801154,1,"LINESTRING(-122.14302957058 47.6376816630363, ..."
572,1147800845,1,"LINESTRING(-122.14302957058 47.6378801465034, ..."
573,1147800842,1,"LINESTRING(-122.14291960001 47.6386311650276, ..."
574,1147800843,1,"LINESTRING(-122.142908871174 47.6404094696045,..."


In [5]:
route_network_df = duckdb.query(
    f"""
    SELECT edge_id, linestring 
    FROM '{newson_krumm_route_network_path}'
    """
).to_df()
route_network_df

,edge_id,linestring
0,883991900000,"LINESTRING(-122.732318937778 47.8899192810059,..."
1,883991900001,"LINESTRING(-122.71107852459 47.8776508569717, ..."
2,883991900002,"LINESTRING(-122.707419991493 47.8761515021324,..."
3,883991900003,"LINESTRING(-122.707419991493 47.8761515021324,..."
4,883991900004,"LINESTRING(-122.715329825878 47.8818699717522,..."
...,...,...
158162,884152400184,"LINESTRING(-121.777908504009 47.4525502324104,..."
158163,884152400185,"LINESTRING(-121.781320273876 47.4532207846642,..."
158164,884152400186,"LINESTRING(-121.784308254719 47.4577805399895,..."
158165,884152400187,"LINESTRING(-121.782489717007 47.4503803253174,..."


In [6]:
def parse_linestring(linestring_series: pd.Series) -> pd.Series:
    track_segs = linestring_series.str.replace(r"^LINESTRING\(|\)$", "", regex=True)
    track_segs = track_segs.str.replace(",", ";").replace(r"\s+", " ", regex=True)
    track_segs = track_segs.str.split(";")
    track_segs = track_segs.apply(
        lambda x: [
            tuple((float(lat), float(lon)))
            for (lon, lat) in (point.split() for point in x)
        ]
    )
    return track_segs

In [7]:
ground_truth_df["track_segs"] = parse_linestring(ground_truth_df["linestring"])
route_network_df["track_segs"] = parse_linestring(route_network_df["linestring"])

In [8]:
gps_df = duckdb.query(f"SELECT * FROM '{gps_data_path}'").to_df()
gps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7531 entries, 0 to 7530
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   recorded_timestamp  7531 non-null   datetime64[us]
 1   lon                 7531 non-null   float64       
 2   lat                 7531 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 176.6 KB


In [9]:
gps_df

,recorded_timestamp,lon,lat
0,2009-01-17 20:27:37,-122.107083,47.667483
1,2009-01-17 20:27:38,-122.107067,47.667500
2,2009-01-17 20:27:39,-122.107067,47.667500
3,2009-01-17 20:27:40,-122.107033,47.667517
4,2009-01-17 20:27:41,-122.106983,47.667533
...,...,...,...
7526,2009-01-17 22:34:24,-122.141917,47.641450
7527,2009-01-17 22:34:25,-122.141783,47.641450
7528,2009-01-17 22:34:26,-122.141650,47.641433
7529,2009-01-17 22:34:27,-122.141517,47.641450


In [10]:
gps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7531 entries, 0 to 7530
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   recorded_timestamp  7531 non-null   datetime64[us]
 1   lon                 7531 non-null   float64       
 2   lat                 7531 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 176.6 KB


In [11]:
import pandas as pd
from pathlib import Path
import requests
from typing import Dict
import json


def request_osrm_map_matching(
    gps_df: pd.DataFrame, 
    output_path: Path,
    osrm_base_url: str = "http://localhost:5000"
) -> Dict:
    """
    Map matching com OSRM retornando way IDs no campo 'name'
    
    Args:
        gps_df: DataFrame com colunas 'lon', 'lat', 'recorded_timestamp'
        output_path: Caminho para salvar resultado JSON
        osrm_base_url: URL base do OSRM
    
    Returns:
        Dict com resultado completo do OSRM
    """
    points = gps_df[["lon", "lat", "recorded_timestamp"]]
    points = points.sort_values("recorded_timestamp")
    
    timestamps_sec = pd.Series(
        [
            int(dt.timestamp())
            for dt in points["recorded_timestamp"].dt.to_pydatetime().tolist()  # type: ignore
        ]
    )

    
    coordinates = ";".join(
        points.apply(lambda row: f"{row['lon']},{row['lat']}", axis=1)
    )
    timestamps = ";".join(timestamps_sec.astype(str))
    
    url = f"{osrm_base_url}/match/v1/car/{coordinates}"
    params = {
        "timestamps": timestamps,
        "annotations": "nodes,distance,duration,speed",
        "overview": "full",
        "geometries": "geojson",
        "steps": "true"  # IMPORTANTE: precisa para ter way IDs no 'name'
    }
    
    print(f"📡 Requisitando OSRM com {len(points)} pontos...")
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    result = response.json()
    
    # Salva resultado
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(result, f, indent=2)
    
    if result.get("code") == "Ok":
        print("✓ Map matching concluído!")
    
    return result


def extract_way_ids(osrm_result: Dict) -> pd.DataFrame:
    """
    Extrai way IDs e metadados do resultado do OSRM
    
    Returns:
        DataFrame com way_id, distance, duration, geometry
    """
    if osrm_result.get("code") != "Ok":
        return pd.DataFrame()
    
    rows = []
    
    for matching_idx, matching in enumerate(osrm_result.get("matchings", [])):
        for leg_idx, leg in enumerate(matching.get("legs", [])):
            for step_idx, step in enumerate(leg.get("steps", [])):
                
                # Way ID está no campo 'name'
                way_id = step.get("name", "")
                
                rows.append({
                    "matching_idx": matching_idx,
                    "leg_idx": leg_idx,
                    "step_idx": step_idx,
                    "way_id": way_id,  # OSM Way ID aqui!
                    "distance": step.get("distance", 0),
                    "duration": step.get("duration", 0),
                    "mode": step.get("mode", "driving"),
                    "maneuver_type": step.get("maneuver", {}).get("type", ""),
                    "geometry": step.get("geometry", {})
                })
    
    df = pd.DataFrame(rows)
    
    if not df.empty:
        print(f"✓ Extraídos {len(df)} segmentos")
        print(f"  - Way IDs únicos: {df['way_id'].nunique()}")
        print(f"  - Distância total: {df['distance'].sum():.2f}m")
    
    return df


def get_way_statistics(way_df: pd.DataFrame) -> pd.DataFrame:
    """Agrupa estatísticas por way ID"""
    if way_df.empty:
        return pd.DataFrame()
    
    stats = way_df.groupby("way_id").agg({
        "distance": "sum",
        "duration": "sum",
        "step_idx": "count"
    }).rename(columns={"step_idx": "count"})
    
    stats["avg_speed_kmh"] = (stats["distance"] / stats["duration"] * 3.6).round(2)
    
    return stats.reset_index().sort_values("distance", ascending=False)


In [12]:
result = request_osrm_map_matching(
    gps_df=gps_df,
    output_path=Path("output/osrm_result.json")
)

way_df = extract_way_ids(result)
stats = get_way_statistics(way_df)

📡 Requisitando OSRM com 7531 pontos...


/tmp/ipykernel_971017/26885707.py:30: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  for dt in points["recorded_timestamp"].dt.to_pydatetime().tolist()  # type: ignore


✓ Map matching concluído!
✓ Extraídos 14905 segmentos
  - Way IDs únicos: 575
  - Distância total: 77685.30m


In [13]:
way_df.head()

,matching_idx,leg_idx,step_idx,way_id,distance,duration,mode,maneuver_type,geometry
0,0,0,0,1147800801,1.1,0.2,driving,depart,"{'coordinates': [[-122.10708, 47.667529], [-12..."
1,0,0,1,1147800801,0.0,0.0,driving,arrive,"{'coordinates': [[-122.107067, 47.667529], [-1..."
2,0,1,0,1147800801,0.0,0.0,driving,depart,"{'coordinates': [[-122.107067, 47.667529], [-1..."
3,0,1,1,1147800801,0.0,0.0,driving,arrive,"{'coordinates': [[-122.107067, 47.667529], [-1..."
4,0,2,0,1147800801,2.6,0.2,driving,depart,"{'coordinates': [[-122.107067, 47.667529], [-1..."


In [14]:
import geopandas as gpd
from shapely.geometry import LineString

way_df["geometry"] = way_df["geometry"].apply(
    lambda geom: LineString([(coord[0], coord[1]) for coord in geom["coordinates"]]) # type: ignore
)

gdf = gpd.GeoDataFrame(way_df, geometry="geometry")
gdf.head()

,matching_idx,leg_idx,step_idx,way_id,distance,duration,mode,maneuver_type,geometry
0,0,0,0,1147800801,1.1,0.2,driving,depart,"LINESTRING (-122.10708 47.66753, -122.10707 47..."
1,0,0,1,1147800801,0.0,0.0,driving,arrive,"LINESTRING (-122.10707 47.66753, -122.10707 47..."
2,0,1,0,1147800801,0.0,0.0,driving,depart,"LINESTRING (-122.10707 47.66753, -122.10707 47..."
3,0,1,1,1147800801,0.0,0.0,driving,arrive,"LINESTRING (-122.10707 47.66753, -122.10707 47..."
4,0,2,0,1147800801,2.6,0.2,driving,depart,"LINESTRING (-122.10707 47.66753, -122.10703 47..."


In [15]:
coords_df = pd.DataFrame(
    {
        "lat": [lat for geom in gdf.geometry for lon, lat in geom.coords],
        "lon": [lon for geom in gdf.geometry for lon, lat in geom.coords]
    }
)
coords_df.head()

,lat,lon
0,47.667529,-122.107080
1,47.667529,-122.107067
2,47.667529,-122.107067
3,47.667529,-122.107067
4,47.667529,-122.107067


In [16]:
way_df["way_id"].unique()

array(['1147800801', '1147800802', '1147800421', '1147800422',
       '1147800423', '1147800805', '1147800804', '1147800806',
       '1147800764', '1147800776', '1147800761', '1147800762',
       '1147800760', '1147800774', '1147801630', '1147801640',
       '1147801639', '1147801628', '1147801632', '1147801618',
       '1147801048', '1147801627', '1147801050', '1147801049',
       '1147801030', '1147801032', '1147801031', '1147801040',
       '1147801041', '1147801033', '1147801035', '1147801038',
       '1147801039', '1147801034', '1147801037', '1147801042',
       '1147801043', '1147801044', '1147801045', '1147801179',
       '1147801178', '1147801176', '1147801256', '1147801260',
       '1147801264', '1147801263', '1147801262', '1147801261',
       '1147801259', '1147801253', '1147801246', '1147801245',
       '1147801257', '1147801258', '1147801244', '1147801248',
       '1147801247', '1147801251', '1147801249', '1147801254',
       '1147801250', '1147801255', '1147801252', '11484

In [17]:
import plotly.express as px

fig = px.line_map(
    coords_df,
    lat="lat",
    lon="lon",
    title="GPS Track"
)
fig.show()

In [18]:
way_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14905 entries, 0 to 14904
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   matching_idx   14905 non-null  int64  
 1   leg_idx        14905 non-null  int64  
 2   step_idx       14905 non-null  int64  
 3   way_id         14905 non-null  object 
 4   distance       14905 non-null  float64
 5   duration       14905 non-null  float64
 6   mode           14905 non-null  object 
 7   maneuver_type  14905 non-null  object 
 8   geometry       14905 non-null  object 
dtypes: float64(2), int64(3), object(4)
memory usage: 1.0+ MB


In [19]:
G = ox.graph_from_xml(GRAPH_PATH)
nodes, edges = ox.graph_to_gdfs(G)

In [20]:
edges.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 264383 entries, (np.int64(3), np.int64(344566), np.int64(0)) to (np.int64(418442), np.int64(152470), np.int64(0))
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   osmid     264383 non-null  object  
 1   highway   264383 non-null  object  
 2   maxspeed  264383 non-null  object  
 3   oneway    264383 non-null  bool    
 4   reversed  264383 non-null  object  
 5   length    264383 non-null  float64 
 6   geometry  264383 non-null  geometry
dtypes: bool(1), float64(1), geometry(1), object(4)
memory usage: 24.3+ MB


In [21]:
edges.head()

osmid       highway maxspeed  oneway  \
u v      key                                                            
3 344566 0                  1010503151  unclassified     40.0   False   
  181835 0                  1010503605  unclassified     21.0   False   
  19491  0    [1010503149, 1010501823]  unclassified     40.0   False   
4 312331 0                  1130401246  unclassified     40.0   False   
  370597 0                  1130401247  unclassified     40.0   False   

                   reversed      length  \
u v      key                              
3 344566 0             True   12.228187   
  181835 0            False   38.796538   
  19491  0    [False, True]  296.720156   
4 312331 0             True   46.209077   
  370597 0            False  139.163476   

                                                       geometry  
u v      key                                                     
3 344566 0    LINESTRING (-122.20972 47.9064, -122.20972 47....  
  181835 0    LINESTRING (-122.20972 47.9064, -122.21024 47....  
  19491  0    LINESTRING (-122.20972 47.9064, -122.20974 47....  
4 312331 0    LINESTRING (-122.66428 47.56038, -122.66488 47...  
  370597 0    LINESTRING (-122.66428 47.56038, -122.66249 47...

In [22]:
from mmlib.types import GPSPoint

gps_points = [
    GPSPoint(lat=row['lat'], lon=row['lon'], time=row['recorded_timestamp'])
    for _, row in gps_df.iterrows()
]

In [23]:
len(gps_points)

7531

In [24]:
from mmlib.matcher import osrm_matcher

matcher = osrm_matcher(base_url=OSRM_BASE_URL)
result = matcher.match(gps_points)
result.plot(zoom=10)

In [25]:
gt_edge_ids = ground_truth_edge_ids=ground_truth_df["edge_id"].astype(str).tolist()
gt_edge_ids[:3]

['1147800801', '1147800802', '1147800421']

In [26]:
mask = edges.osmid.astype(str).isin(gt_edge_ids)
edges[mask].osmid

u       v       key
1025    310521  0      1148005874
        283078  0      1148005876
1654    295658  0      1130303045
3006    113028  0      1147801055
        249469  0      1147801055
                          ...    
411551  243903  0      1130900142
        340172  0      1130900143
412578  194183  0      1147800892
416659  326929  0      1130901101
418118  109805  0      1130304293
Name: osmid, Length: 556, dtype: object

In [ ]:
result.plot_on_folium(G, gt_edge_ids).show_in_browser()

Your map should have been opened in your browser automatically.
Press ctrl+c to return.


kf.service.services: KApplicationTrader: mimeType "x-scheme-handler/file" not found
[2] Sandbox: CanCreateUserNamespace() clone() failure: EPERM
